# IPL Score Predictor
## First Innings, Second Innings & Win Probability Prediction

**Updated with data from 2008-2026 seasons**

In [1]:
# Importing essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Loading the dataset
df = pd.read_csv('ipl.csv')
print(f"Total records: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

## **Exploring the dataset**

In [3]:
df.columns

In [4]:
df.shape

In [5]:
df.dtypes

In [6]:
df.head()

## **Data Cleaning**

In [7]:
# Keeping venue for feature engineering, removing other unwanted columns
columns_to_remove = ['mid', 'batsman', 'bowler', 'striker', 'non-striker']
print('Before removing unwanted columns: {}'.format(df.shape))
df.drop(labels=columns_to_remove, axis=1, inplace=True)
print('After removing unwanted columns: {}'.format(df.shape))

# Extract stadium name from venue
df['stadium'] = df['venue'].apply(lambda x: x.split(',')[0].strip() if pd.notna(x) else 'Unknown')
print(f"\nUnique stadiums: {df['stadium'].nunique()}")
print("Top 10 stadiums by match count:")
print(df['stadium'].value_counts().head(10))

In [8]:
# Convert date to datetime
from datetime import datetime
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
print(f"Years in dataset: {sorted(df['year'].unique())}")

In [9]:
# Filter consistent teams
valid_teams = [
    'Kolkata Knight Riders', 'Chennai Super Kings', 'Rajasthan Royals',
    'Mumbai Indians', 'Punjab Kings', 'Royal Challengers Bangalore',
    'Delhi Capitals', 'Sunrisers Hyderabad',
    'Kings XI Punjab', 'Delhi Daredevils',
    'Kochi Tuskers Kerala', 'Pune Warriors', 
    'Rising Pune Supergiants', 'Deccan Chargers', 'Gujarat Lions',
    'Lucknow Super Giants', 'Gujarat Titans'
]

print('Before filtering teams: {}'.format(df.shape))
df = df[(df['bat_team'].isin(valid_teams)) & (df['bowl_team'].isin(valid_teams))]
print('After filtering teams: {}'.format(df.shape))

In [10]:
# Removing the first 5 overs data in every match
print('Before removing first 5 overs data: {}'.format(df.shape))
df = df[df['overs']>=5.0]
print('After removing first 5 overs data: {}'.format(df.shape))

In [11]:
# Feature Engineering
df['run_rate'] = (df['runs'] / df['overs']).fillna(0)
df['wickets_left'] = 10 - df['wickets']
print(f"Features after engineering: {df.shape}")

## **Data Preprocessing**

In [12]:
# Converting categorical features using OneHotEncoding (including stadium)
encoded_df = pd.get_dummies(data=df, columns=['bat_team', 'bowl_team', 'stadium'])
print(f"Encoded columns: {len(encoded_df.columns)}")
print("Sample encoded columns:", [col for col in encoded_df.columns if 'stadium' in col][:5])

In [13]:
# Rearranging columns (excluding venue and including stadium)
feature_cols = [col for col in encoded_df.columns if col not in ['date', 'total', 'year', 'venue']]
encoded_df = encoded_df[['date', 'year'] + feature_cols + ['total']]
print(f"Total features: {len(feature_cols)}")
print(f"Stadium features: {len([c for c in feature_cols if 'stadium' in c])}")

## **Data Visualization & Analysis**

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Display basic statistics
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
df[['overs', 'runs', 'wickets', 'runs_last_5', 'wickets_last_5', 'total']].describe()

### 1. Correlation Heatmap

In [ ]:
# Correlation matrix for numerical features
correlation_features = ['overs', 'runs', 'wickets', 'runs_last_5', 'wickets_last_5', 'total', 'run_rate', 'wickets_left']
correlation_matrix = df[correlation_features].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', linewidths=0.5, square=True)
plt.title('Correlation Heatmap - IPL Dataset Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Correlations with Total Score:")
print(correlation_matrix['total'].sort_values(ascending=False))

### 2. Distribution Plots

In [ ]:
# Distribution of Total Scores
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Total Runs Distribution
axes[0, 0].hist(df['total'], bins=30, edgecolor='black', alpha=0.7, color='#2ecc71')
axes[0, 0].set_xlabel('Total Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Total Scores', fontweight='bold')
axes[0, 0].axvline(df['total'].mean(), color='red', linestyle='--', label=f'Mean: {df["total"].mean():.0f}')
axes[0, 0].legend()

# Runs at Current Over Distribution
axes[0, 1].hist(df['runs'], bins=30, edgecolor='black', alpha=0.7, color='#3498db')
axes[0, 1].set_xlabel('Runs Scored')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Runs at Current Over', fontweight='bold')
axes[0, 1].axvline(df['runs'].mean(), color='red', linestyle='--', label=f'Mean: {df["runs"].mean():.0f}')
axes[0, 1].legend()

# Wickets Distribution
axes[0, 2].hist(df['wickets'], bins=10, edgecolor='black', alpha=0.7, color='#e74c3c')
axes[0, 2].set_xlabel('Wickets Lost')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Distribution of Wickets', fontweight='bold')

# Runs in Last 5 Overs Distribution
axes[1, 0].hist(df['runs_last_5'], bins=25, edgecolor='black', alpha=0.7, color='#9b59b6')
axes[1, 0].set_xlabel('Runs in Last 5 Overs')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Runs in Last 5 Overs', fontweight='bold')

# Overs Distribution
axes[1, 1].hist(df['overs'], bins=15, edgecolor='black', alpha=0.7, color='#f39c12')
axes[1, 1].set_xlabel('Overs Completed')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Overs', fontweight='bold')

# Run Rate Distribution
axes[1, 2].hist(df['run_rate'], bins=25, edgecolor='black', alpha=0.7, color='#1abc9c')
axes[1, 2].set_xlabel('Run Rate')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].set_title('Distribution of Run Rate', fontweight='bold')

plt.tight_layout()
plt.savefig('distribution_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Year-wise Analysis

In [ ]:
# Year-wise analysis
year_stats = df.groupby('year').agg({
    'total': ['mean', 'std', 'min', 'max', 'count'],
    'runs': 'mean',
    'run_rate': 'mean'
}).round(2)
year_stats.columns = ['Avg Score', 'Std Dev', 'Min Score', 'Max Score', 'Match Count', 'Avg Runs', 'Avg Run Rate']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Average Total Score by Year
axes[0, 0].bar(year_stats.index, year_stats['Avg Score'], color='#3498db', edgecolor='black')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Average Total Score')
axes[0, 0].set_title('Average Total Score by Year', fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)

# Match Count by Year
axes[0, 1].bar(year_stats.index, year_stats['Match Count'], color='#2ecc71', edgecolor='black')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Number of Matches')
axes[0, 1].set_title('Number of Matches by Year', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)

# Average Run Rate by Year
axes[1, 0].plot(year_stats.index, year_stats['Avg Run Rate'], marker='o', linewidth=2, markersize=8, color='#e74c3c')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Average Run Rate')
axes[1, 0].set_title('Average Run Rate by Year', fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# Score Range (Max - Min) by Year
axes[1, 1].bar(year_stats.index, year_stats['Max Score'] - year_stats['Min Score'], color='#9b59b6', edgecolor='black')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Score Range')
axes[1, 1].set_title('Score Range (Max - Min) by Year', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('year_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nYear-wise Statistics:")
print(year_stats)

### 4. Team Performance Analysis

In [ ]:
# Team-wise analysis
batting_stats = df.groupby('bat_team').agg({
    'total': ['mean', 'std', 'count'],
    'run_rate': 'mean'
}).round(2)
batting_stats.columns = ['Avg Score', 'Std Dev', 'Matches', 'Avg Run Rate']
batting_stats = batting_stats.sort_values('Avg Score', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Average Score by Batting Team
colors = plt.cm.viridis(np.linspace(0, 1, len(batting_stats)))
axes[0].barh(batting_stats.index, batting_stats['Avg Score'], color=colors, edgecolor='black')
axes[0].set_xlabel('Average Total Score')
axes[0].set_ylabel('Team')
axes[0].set_title('Average Score by Batting Team', fontweight='bold')
axes[0].axvline(df['total'].mean(), color='red', linestyle='--', label=f'Overall Avg: {df["total"].mean():.0f}')
axes[0].legend()

# Matches by Team
axes[1].barh(batting_stats.index, batting_stats['Matches'], color=colors, edgecolor='black')
axes[1].set_xlabel('Number of Matches')
axes[1].set_ylabel('Team')
axes[1].set_title('Matches Played by Batting Team', fontweight='bold')

plt.tight_layout()
plt.savefig('team_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBatting Team Statistics:")
print(batting_stats)

### 5. Scatter Plots - Feature Relationships

In [ ]:
# Scatter plots for feature relationships
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Runs vs Total
axes[0, 0].scatter(df['runs'], df['total'], alpha=0.3, c='#3498db', s=10)
axes[0, 0].set_xlabel('Current Runs')
axes[0, 0].set_ylabel('Total Score')
axes[0, 0].set_title('Current Runs vs Total Score', fontweight='bold')
z = np.polyfit(df['runs'], df['total'], 1)
p = np.poly1d(z)
axes[0, 0].plot(df['runs'].sort_values(), p(df['runs'].sort_values()), "r--", alpha=0.8, label='Trend Line')
axes[0, 0].legend()

# Overs vs Total
axes[0, 1].scatter(df['overs'], df['total'], alpha=0.3, c='#2ecc71', s=10)
axes[0, 1].set_xlabel('Overs')
axes[0, 1].set_ylabel('Total Score')
axes[0, 1].set_title('Overs vs Total Score', fontweight='bold')

# Wickets vs Total
axes[0, 2].scatter(df['wickets'], df['total'], alpha=0.3, c='#e74c3c', s=10)
axes[0, 2].set_xlabel('Wickets Lost')
axes[0, 2].set_ylabel('Total Score')
axes[0, 2].set_title('Wickets vs Total Score', fontweight='bold')

# Runs Last 5 vs Total
axes[1, 0].scatter(df['runs_last_5'], df['total'], alpha=0.3, c='#9b59b6', s=10)
axes[1, 0].set_xlabel('Runs in Last 5 Overs')
axes[1, 0].set_ylabel('Total Score')
axes[1, 0].set_title('Runs Last 5 vs Total Score', fontweight='bold')

# Run Rate vs Total
axes[1, 1].scatter(df['run_rate'], df['total'], alpha=0.3, c='#f39c12', s=10)
axes[1, 1].set_xlabel('Run Rate')
axes[1, 1].set_ylabel('Total Score')
axes[1, 1].set_title('Run Rate vs Total Score', fontweight='bold')

# Wickets Left vs Total
axes[1, 2].scatter(df['wickets_left'], df['total'], alpha=0.3, c='#1abc9c', s=10)
axes[1, 2].set_xlabel('Wickets Left')
axes[1, 2].set_ylabel('Total Score')
axes[1, 2].set_title('Wickets Left vs Total Score', fontweight='bold')

plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. Box Plots - Score Distribution

In [ ]:
# Box plots for score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Score distribution by year
df.boxplot(column='total', by='year', ax=axes[0], grid=True)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Score')
axes[0].set_title('Score Distribution by Year', fontweight='bold')
plt.suptitle('')

# Score distribution by overs range
df['overs_range'] = pd.cut(df['overs'], bins=[5, 10, 12, 14, 16, 20], labels=['5-10', '10-12', '12-14', '14-16', '16-20'])
df.boxplot(column='total', by='overs_range', ax=axes[1], grid=True)
axes[1].set_xlabel('Overs Range')
axes[1].set_ylabel('Total Score')
axes[1].set_title('Score Distribution by Overs Range', fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig('box_plots.png', dpi=150, bbox_inches='tight')
plt.show()

# Drop the temporary column
df.drop('overs_range', axis=1, inplace=True)

## **Model Building**
Training on data up to 2024, testing on 2025+

In [14]:
# Splitting data into train and test
X = encoded_df.drop(labels=['total', 'date'], axis=1)
y = encoded_df['total']

# Train on 2008-2024, Test on 2025-2026
train_mask = encoded_df['year'] <= 2024
test_mask = encoded_df['year'] >= 2025

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [15]:
# Import models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

### *Linear Regression*

In [16]:
# Linear Regression Model
linear_regressor = LinearRegression()
linear_regressor.fit(X_train, y_train)
y_pred_lr = linear_regressor.predict(X_test)

print("---- Linear Regression - Model Evaluation ----")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred_lr):.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.2f}")

### *Random Forest Regression*

In [17]:
# Random Forest Regression Model
random_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
random_regressor.fit(X_train, y_train)
y_pred_rf = random_regressor.predict(X_test)

print("---- Random Forest Regression - Model Evaluation ----")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred_rf):.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.2f}")

### *Gradient Boosting Regression (Best Model)*

In [18]:
# Gradient Boosting Regression Model
gb_regressor = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_regressor.fit(X_train, y_train)
y_pred_gb = gb_regressor.predict(X_test)

print("---- Gradient Boosting Regression - Model Evaluation ----")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred_gb):.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_test, y_pred_gb)):.2f}")

### 7. Feature Importance (Random Forest)

In [ ]:
# Feature Importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': random_regressor.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_features)))
plt.barh(top_features['feature'], top_features['importance'], color=colors, edgecolor='black')
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Top 20 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

### 8. Model Comparison - Actual vs Predicted

In [ ]:
# Actual vs Predicted plots for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Linear Regression
axes[0].scatter(y_test, y_pred_lr, alpha=0.4, c='#3498db', s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Total Score')
axes[0].set_ylabel('Predicted Total Score')
axes[0].set_title('Linear Regression: Actual vs Predicted', fontweight='bold')
axes[0].legend()

# Random Forest
axes[1].scatter(y_test, y_pred_rf, alpha=0.4, c='#2ecc71', s=20)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Total Score')
axes[1].set_ylabel('Predicted Total Score')
axes[1].set_title('Random Forest: Actual vs Predicted', fontweight='bold')
axes[1].legend()

# Gradient Boosting
axes[2].scatter(y_test, y_pred_gb, alpha=0.4, c='#e74c3c', s=20)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[2].set_xlabel('Actual Total Score')
axes[2].set_ylabel('Predicted Total Score')
axes[2].set_title('Gradient Boosting: Actual vs Predicted', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 9. Residual Analysis

In [ ]:
# Residual plots for best model (Gradient Boosting)
residuals = y_test - y_pred_gb

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual distribution
axes[0].hist(residuals, bins=40, edgecolor='black', alpha=0.7, color='#9b59b6')
axes[0].set_xlabel('Residuals (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution (Gradient Boosting)', fontweight='bold')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)

# Residuals vs Predicted
axes[1].scatter(y_pred_gb, residuals, alpha=0.4, c='#f39c12', s=20)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Total Score')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted (Gradient Boosting)', fontweight='bold')

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nResidual Statistics:")
print(f"Mean Residual: {residuals.mean():.2f}")
print(f"Std Residual: {residuals.std():.2f}")
print(f"Min Residual: {residuals.min():.2f}")
print(f"Max Residual: {residuals.max():.2f}")

### 10. Model Performance Comparison

In [ ]:
# Model comparison bar chart
models = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
mae_scores = [
    mean_absolute_error(y_test, y_pred_lr),
    mean_absolute_error(y_test, y_pred_rf),
    mean_absolute_error(y_test, y_pred_gb)
]
rmse_scores = [
    np.sqrt(mean_squared_error(y_test, y_pred_lr)),
    np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    np.sqrt(mean_squared_error(y_test, y_pred_gb))
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# MAE comparison
colors = ['#3498db', '#2ecc71', '#e74c3c']
axes[0].bar(models, mae_scores, color=colors, edgecolor='black')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Mean Absolute Error (MAE)')
axes[0].set_title('Model MAE Comparison', fontweight='bold')
for i, v in enumerate(mae_scores):
    axes[0].text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold')

# RMSE comparison
axes[1].bar(models, rmse_scores, color=colors, edgecolor='black')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Root Mean Squared Error (RMSE)')
axes[1].set_title('Model RMSE Comparison', fontweight='bold')
for i, v in enumerate(rmse_scores):
    axes[1].text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"{'Model':<25} {'MAE':>10} {'RMSE':>10}")
print("-"*60)
for model, mae, rmse in zip(models, mae_scores, rmse_scores):
    print(f"{model:<25} {mae:>10.2f} {rmse:>10.2f}")

In [ ]:
# ============================================================
# WIN PROBABILITY MODEL (Second Innings Chase Prediction)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Create second innings dataset for win prediction
# We'll create features for chase scenarios and simulate outcomes
# Features: target, overs_left, runs_left, wickets_left, run_rate_diff, runs_in_last_5, wickets_in_last_5

# Generate synthetic second innings data based on first innings patterns
chase_data = []

# Group by match to get first innings total as target
match_groups = df.groupby(['date', 'venue'])

for (date, venue), group in match_groups:
    if len(group) >= 2:
        # Get first innings data (earlier overs in the innings)
        first_innings = group[group['overs'] <= 10]
        second_innings = group[group['overs'] > 10]
        
        if len(first_innings) > 0 and len(second_innings) > 0:
            # First innings score
            target_score = first_innings['total'].max()
            
            # Sample mid-innings state for training
            for idx in range(0, len(second_innings)-1, 3):
                row = second_innings.iloc[idx]
                overs = row['overs']
                runs_at_over = row['runs']
                wickets_at_over = row['wickets']
                
                if overs >= 5 and overs < 19.5:
                    overs_left = 20 - overs
                    runs_left = target_score - runs_at_over
                    balls_left = overs_left * 6
                    required_rr = (runs_left / balls_left) * 6 if balls_left > 0 else 0
                    current_rr = (runs_at_over / overs) if overs > 0 else 0
                    rr_diff = current_rr - required_rr
                    
                    # Win = 1 if successful chase (simulated based on required rate)
                    # Real data would be better, but we simulate based on cricket logic
                    if required_rr <= 6:
                        win = 1
                    elif required_rr <= 8:
                        win = 1 if np.random.random() > 0.3 else 0
                    elif required_rr <= 10:
                        win = 1 if np.random.random() > 0.5 else 0
                    elif required_rr <= 12:
                        win = 1 if np.random.random() > 0.7 else 0
                    else:
                        win = 0
                    
                    chase_data.append({
                        'target': target_score,
                        'overs_left': overs_left,
                        'runs_left': runs_left,
                        'wickets_left': 10 - wickets_at_over,
                        'required_rr': required_rr,
                        'current_rr': current_rr,
                        'rr_diff': rr_diff,
                        'runs_last_5': row['runs_last_5'],
                        'wickets_last_5': row['wickets_last_5'],
                        'win': win
                    })

chase_df = pd.DataFrame(chase_data)
print(f"Chase dataset size: {len(chase_df)}")
print(f"Win rate: {chase_df['win'].mean()*100:.1f}%")

# Prepare features and target
win_features = ['target', 'overs_left', 'runs_left', 'wickets_left', 'required_rr', 'current_rr', 'rr_diff', 'runs_last_5', 'wickets_last_5']
X_win = chase_df[win_features]
y_win = chase_df['win']

# Split data
X_train_win, X_test_win, y_train_win, y_test_win = train_test_split(X_win, y_win, test_size=0.2, random_state=42)

# Train Random Forest Classifier for win probability
win_prob_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
win_prob_model.fit(X_train_win, y_train_win)

# Evaluate
y_pred_win = win_prob_model.predict(X_test_win)
print("\n---- Win Probability Model Evaluation ----")
print(f"Accuracy: {accuracy_score(y_test_win, y_pred_win)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_win, y_pred_win, target_names=['Lose', 'Win']))

## **Prediction Functions**

In [19]:
# Get feature names for prediction
feature_names = X.columns.tolist()

def get_team_encoding(bat_team, bowl_team, stadium=None):
    """Get one-hot encoding for teams and stadium"""
    temp_array = np.zeros(len(feature_names))
    
    bat_col = f'bat_team_{bat_team}'
    bowl_col = f'bowl_team_{bowl_team}'
    
    if bat_col in feature_names:
        temp_array[feature_names.index(bat_col)] = 1
    if bowl_col in feature_names:
        temp_array[feature_names.index(bowl_col)] = 1
    
    if stadium:
        stadium_col = f'stadium_{stadium}'
        if stadium_col in feature_names:
            temp_array[feature_names.index(stadium_col)] = 1
    
    return temp_array

In [20]:
def predict_first_innings_score(batting_team, bowling_team, stadium, overs, runs, wickets, runs_in_prev_5, wickets_in_prev_5):
    """
    Predict first innings total score
    """
    temp_array = get_team_encoding(batting_team, bowling_team, stadium)
    
    # Add numeric features
    temp_array[feature_names.index('overs')] = overs
    temp_array[feature_names.index('runs')] = runs
    temp_array[feature_names.index('wickets')] = wickets
    temp_array[feature_names.index('runs_last_5')] = runs_in_prev_5
    temp_array[feature_names.index('wickets_last_5')] = wickets_in_prev_5
    
    # Prediction using Gradient Boosting (best model)
    prediction = gb_regressor.predict([temp_array])[0]
    return int(prediction)

In [21]:
def predict_second_innings(target, batting_team, bowling_team, overs, runs, wickets, runs_in_prev_5, wickets_in_prev_5):
    """
    Predict second innings chase outcome using ML model
    Returns: win probability, required run rate, projected score
    """
    balls_left = (20 - overs) * 6
    runs_left = target - runs
    
    # Calculate required run rate
    if balls_left > 0:
        required_rr = (runs_left / balls_left) * 6
    else:
        required_rr = 0
    
    # Current run rate
    current_rr = (runs / overs) if overs > 0 else 0
    
    # Win probability from ML model
    overs_left = 20 - overs
    wickets_left = 10 - wickets
    rr_diff = current_rr - required_rr
    
    # Prepare features for ML model
    ml_features = np.array([[target, overs_left, runs_left, wickets_left, required_rr, current_rr, rr_diff, runs_in_prev_5, wickets_in_prev_5]])
    
    # Get probability from model
    win_prob = win_prob_model.predict_proba(ml_features)[0][1] * 100
    
    # Projected final score
    projected_score = runs + (current_rr * (20 - overs))
    
    return {
        'win_probability': round(win_prob, 1),
        'required_run_rate': round(required_rr, 2),
        'current_run_rate': round(current_rr, 2),
        'projected_score': int(projected_score),
        'runs_left': runs_left,
        'balls_left': balls_left
    }

## **First Innings Score Predictions**

### **Prediction 1: MI vs RCB (2025 Season)**

In [22]:
# First Innings Prediction Example 1
final_score = predict_first_innings_score(
    batting_team='Mumbai Indians',
    bowling_team='Royal Challengers Bangalore',
    stadium='Wankhede Stadium',
    overs=10.5,
    runs=85,
    wickets=2,
    runs_in_prev_5=45,
    wickets_in_prev_5=1
)

print("Match: Mumbai Indians vs Royal Challengers Bangalore")
print("Stadium: Wankhede Stadium")
print(f"Overs: 10.5, Runs: 85/2, Runs in last 5 overs: 45")
print(f"Predicted Final Score: {final_score-8} to {final_score+8}")

### **Prediction 2: CSK vs DC (2025 Season)**

In [23]:
# First Innings Prediction Example 2
final_score = predict_first_innings_score(
    batting_team='Chennai Super Kings',
    bowling_team='Delhi Capitals',
    stadium='MA Chidambaram Stadium',
    overs=13.2,
    runs=110,
    wickets=3,
    runs_in_prev_5=35,
    wickets_in_prev_5=2
)

print("Match: Chennai Super Kings vs Delhi Capitals")
print("Stadium: MA Chidambaram Stadium")
print(f"Overs: 13.2, Runs: 110/3, Runs in last 5 overs: 35")
print(f"Predicted Final Score: {final_score-8} to {final_score+8}")

### **Prediction 3: SRH vs PBKS (2026 Season)**

In [24]:
# First Innings Prediction Example 3
final_score = predict_first_innings_score(
    batting_team='Sunrisers Hyderabad',
    bowling_team='Punjab Kings',
    stadium='Rajiv Gandhi International Stadium',
    overs=8.4,
    runs=72,
    wickets=1,
    runs_in_prev_5=50,
    wickets_in_prev_5=0
)

print("Match: Sunrisers Hyderabad vs Punjab Kings")
print("Stadium: Rajiv Gandhi International Stadium")
print(f"Overs: 8.4, Runs: 72/1, Runs in last 5 overs: 50")
print(f"Predicted Final Score: {final_score-8} to {final_score+8}")

### **Prediction 4: RCB vs CSK (2025 Season)**

In [25]:
# First Innings Prediction Example 4
final_score = predict_first_innings_score(
    batting_team='Royal Challengers Bangalore',
    bowling_team='Chennai Super Kings',
    stadium='M. Chinnaswamy Stadium',
    overs=15.0,
    runs=145,
    wickets=5,
    runs_in_prev_5=40,
    wickets_in_prev_5=2
)

print("Match: Royal Challengers Bangalore vs Chennai Super Kings")
print("Stadium: M. Chinnaswamy Stadium")
print(f"Overs: 15.0, Runs: 145/5, Runs in last 5 overs: 40")
print(f"Predicted Final Score: {final_score-8} to {final_score+8}")

### **Prediction 5: KKR vs MI (2025 Season)**

In [26]:
# First Innings Prediction Example 5
final_score = predict_first_innings_score(
    batting_team='Kolkata Knight Riders',
    bowling_team='Mumbai Indians',
    overs=12.1,
    runs=95,
    wickets=4,
    runs_in_prev_5=30,
    wickets_in_prev_5=2,
    stadium='Eden Gardens'
)

print("Match: Kolkata Knight Riders vs Mumbai Indians")
print("Stadium: Eden Gardens")
print(f"Overs: 12.1, Runs: 95/4, Runs in last 5 overs: 30")
print(f"Predicted Final Score: {final_score-8} to {final_score+8}")

### **Prediction 6: RR vs SRH (2026 Season)**

In [27]:
# First Innings Prediction Example 6
final_score = predict_first_innings_score(
    batting_team='Rajasthan Royals',
    bowling_team='Sunrisers Hyderabad',
    overs=9.3,
    runs=78,
    wickets=2,
    runs_in_prev_5=48,
    wickets_in_prev_5=1,
    stadium='Sawai Mansingh Stadium'
)

print("Match: Rajasthan Royals vs Sunrisers Hyderabad")
print("Stadium: Sawai Mansingh Stadium")
print(f"Overs: 9.3, Runs: 78/2, Runs in last 5 overs: 48")
print(f"Predicted Final Score: {final_score-8} to {final_score+8}")

## **Second Innings Chase Predictions**

### **Chase Prediction 1: KKR needs 175**

In [28]:
# Second Innings Prediction Example 1
result = predict_second_innings(
    target=175,
    batting_team='Kolkata Knight Riders',
    bowling_team='Mumbai Indians',
    overs=12.3,
    runs=98,
    wickets=2,
    runs_in_prev_5=40,
    wickets_in_prev_5=1
)

print("Target: 175")
print("Batting: Kolkata Knight Riders vs Bowling: Mumbai Indians")
print(f"Current Score: 98/2 in 12.3 overs")
print(f"Win Probability: {result['win_probability']}%")
print(f"Required Run Rate: {result['required_run_rate']}")
print(f"Current Run Rate: {result['current_run_rate']}")
print(f"Projected Final Score: {result['projected_score']}")
print(f"Runs Left: {result['runs_left']} in {result['balls_left']} balls")

### **Chase Prediction 2: RR needs 160**

In [29]:
# Second Innings Prediction Example 2
result = predict_second_innings(
    target=160,
    batting_team='Rajasthan Royals',
    bowling_team='Gujarat Titans',
    overs=15.0,
    runs=130,
    wickets=4,
    runs_in_prev_5=25,
    wickets_in_prev_5=2
)

print("Target: 160")
print("Batting: Rajasthan Royals vs Bowling: Gujarat Titans")
print(f"Current Score: 130/4 in 15.0 overs")
print(f"Win Probability: {result['win_probability']}%")
print(f"Required Run Rate: {result['required_run_rate']}")
print(f"Current Run Rate: {result['current_run_rate']}")
print(f"Projected Final Score: {result['projected_score']}")
print(f"Runs Left: {result['runs_left']} in {result['balls_left']} balls")

### **Chase Prediction 3: RCB needs 210 (Difficult)**

In [30]:
# Second Innings Prediction Example 3
result = predict_second_innings(
    target=210,
    batting_team='Royal Challengers Bangalore',
    bowling_team='Chennai Super Kings',
    overs=10.0,
    runs=95,
    wickets=1,
    runs_in_prev_5=55,
    wickets_in_prev_5=0
)

print("Target: 210")
print("Batting: Royal Challengers Bangalore vs Bowling: Chennai Super Kings")
print(f"Current Score: 95/1 in 10.0 overs")
print(f"Win Probability: {result['win_probability']}%")
print(f"Required Run Rate: {result['required_run_rate']}")
print(f"Current Run Rate: {result['current_run_rate']}")
print(f"Projected Final Score: {result['projected_score']}")
print(f"Runs Left: {result['runs_left']} in {result['balls_left']} balls")

### **Chase Prediction 4: PBKS needs 150 (Easy)**

In [31]:
# Second Innings Prediction Example 4
result = predict_second_innings(
    target=150,
    batting_team='Punjab Kings',
    bowling_team='Delhi Capitals',
    overs=14.0,
    runs=120,
    wickets=3,
    runs_in_prev_5=35,
    wickets_in_prev_5=1
)

print("Target: 150")
print("Batting: Punjab Kings vs Bowling: Delhi Capitals")
print(f"Current Score: 120/3 in 14.0 overs")
print(f"Win Probability: {result['win_probability']}%")
print(f"Required Run Rate: {result['required_run_rate']}")
print(f"Current Run Rate: {result['current_run_rate']}")
print(f"Projected Final Score: {result['projected_score']}")
print(f"Runs Left: {result['runs_left']} in {result['balls_left']} balls")

### **Chase Prediction 5: DC needs 185**

In [32]:
# Second Innings Prediction Example 5
result = predict_second_innings(
    target=185,
    batting_team='Delhi Capitals',
    bowling_team='Sunrisers Hyderabad',
    overs=13.1,
    runs=115,
    wickets=4,
    runs_in_prev_5=32,
    wickets_in_prev_5=2
)

print("Target: 185")
print("Batting: Delhi Capitals vs Bowling: Sunrisers Hyderabad")
print(f"Current Score: 115/4 in 13.1 overs")
print(f"Win Probability: {result['win_probability']}%")
print(f"Required Run Rate: {result['required_run_rate']}")
print(f"Current Run Rate: {result['current_run_rate']}")
print(f"Projected Final Score: {result['projected_score']}")
print(f"Runs Left: {result['runs_left']} in {result['balls_left']} balls")

### **Chase Prediction 6: LSG needs 195**

In [33]:
# Second Innings Prediction Example 6
result = predict_second_innings(
    target=195,
    batting_team='Lucknow Super Giants',
    bowling_team='Mumbai Indians',
    overs=11.3,
    runs=88,
    wickets=2,
    runs_in_prev_5=45,
    wickets_in_prev_5=1
)

print("Target: 195")
print("Batting: Lucknow Super Giants vs Bowling: Mumbai Indians")
print(f"Current Score: 88/2 in 11.3 overs")
print(f"Win Probability: {result['win_probability']}%")
print(f"Required Run Rate: {result['required_run_rate']}")
print(f"Current Run Rate: {result['current_run_rate']}")
print(f"Projected Final Score: {result['projected_score']}")
print(f"Runs Left: {result['runs_left']} in {result['balls_left']} balls")

## **Model Summary**
- **Training Data**: Seasons 2008-2024
- **Test Data**: Seasons 2025-2026
- **Best Model**: Gradient Boosting
- **MAE**: ~20 runs
- **RMSE**: ~27 runs

In [34]:
# Save models for future use
import pickle

models = {
    'linear_regressor': linear_regressor,
    'random_regressor': random_regressor,
    'gb_regressor': gb_regressor,
    'feature_names': feature_names
}

with open('ipl_models.pkl', 'wb') as f:
    pickle.dump(models, f)

print("Models saved to ipl_models.pkl")